In [ ]:
import requests
import pandas as pd

In [ ]:
headers = {
    'authoriy' : 'https://wiki.openstreetmap.org/'
}
response = requests.get('https://wiki.openstreetmap.org/wiki/List_of_London_Underground_stations', headers=headers)

# checking the status code and obtaining the html response content
if(response.status_code == 200):
    html_content = response.text

In [ ]:
if(response.status_code == 200):
    html_content = response.text

In [ ]:
all_tables = pd.read_html(html_content)

In [ ]:
underground_df = all_tables[0]

underground_df.columns.format()

In [ ]:
underground_df = underground_df.drop(['Platform / Entrance',
 'Collected By',
 'Collected On',
 'Line',
 'Step free'], axis=1)

In [ ]:
underground_df = underground_df.drop_duplicates(subset=['Name'], keep='first')

In [ ]:
underground_df.tail(60)

In [ ]:
dlr_df = all_tables[1]

In [ ]:
dlr_df.columns.format()

In [ ]:
dlr_df = dlr_df.drop(['Platform / Entrance',
 'Collected By',
 'Collected On'], axis=1)

dlr_df

In [ ]:
dlr_df['Name'].nunique()

In [ ]:
dlr_df = dlr_df.drop_duplicates(subset=['Name'], keep='first')

In [ ]:
dlr_df = dlr_df[dlr_df['Name'] != 'Bank']

In [ ]:
dlr_df

In [ ]:
tfl_df = pd.concat([underground_df, dlr_df], axis=0, ignore_index=True)
tfl_df.head(60)

In [ ]:
import re
def clean_coordinates(df):
    def clean_coordinate(value):
        if isinstance(value, str):
            # Extract the main coordinate value before any +/- or parentheses
            match = re.search(r"^([-+]?\d*\.\d+|\d+)", value)
            if match:
                return float(match.group(1))
        elif isinstance(value, (float, int)):
            return value
        return None
    
    # Apply the cleaning function to Latitude and Longitude columns
    df['Latitude_cleaned'] = df['Latitude'].apply(clean_coordinate)
    df['Longitude_cleaned'] = df['Longitude'].apply(clean_coordinate)
    
    # Identify rows with invalid coordinates
    invalid_rows = df[df['Latitude_cleaned'].isna() | df['Longitude_cleaned'].isna()]
    
    # Remove rows with invalid coordinates from the main DataFrame
    cleaned_df = df.dropna(subset=['Latitude_cleaned', 'Longitude_cleaned'])
    
    # Drop the temporary cleaned columns used for validation
    cleaned_df = cleaned_df.drop(columns=['Latitude_cleaned', 'Longitude_cleaned'])
    invalid_rows = invalid_rows.drop(columns=['Latitude_cleaned', 'Longitude_cleaned'])
    
    return cleaned_df, invalid_rows

In [ ]:
tfl_df_c, tfl_df_invalid = clean_coordinates(tfl_df)

In [ ]:
tfl_df.to_csv('../../data/stations-with-web-coordinates.csv', sep=',', encoding='utf-8', index=False, header=True)